In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
poly_homopolymer_regions = [309,310,311,16179,16180,16181,16182,16183]

In [ ]:
## read in mitoscope qc files
df_pb = pd.read_csv("../hapmap/pacbio/output/qc_summary.tsv", sep='\t')
df_pb[['Donor', 'Seq_Tech', 'Center']] = df_pb['Sample'].str.split('-', expand=True)

df_ont = pd.read_csv("../hapmap/ont/output/qc_summary.tsv", sep='\t')
df_ont[['Donor', 'Seq_Tech', 'Center']] = df_ont['Sample'].str.split('-', expand=True)

df = pd.concat([df_pb, df_ont])
# df = df.sort_values(['Tissue', 'Donor', 'Center'])

df['Seq_Tech'] = np.where(df['Seq_Tech'] == 'pacbio', 'PacBio', 'ONT')
df

In [ ]:
## plot correlation of CN against median read length
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

common_kws = dict(scatter_kws={'s': 40, 'alpha': 0.7, 'edgecolor': 'black'},
                  line_kws={'color': 'red', 'lw': 2})

sns.regplot(df, x='mtDNA_CN', y='Median_Read_Length', ci=None, **common_kws)
plt.xlabel('Mitochondrial Copy Number')
plt.ylabel('Median Read Length')
plt.savefig(f"plots/suppl/mtdna_cn_vs_med_read_length.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## read in SR coverage stats and combine with LR df
comb_sr_cov = pd.DataFrame(columns=['Sample', 'Mito_Coverage'])
sr_samples = ['hapmap-illumina-bcm', 'hapmap-illumina-broad', 'hapmap-illumina-nygc', 'hapmap-illumina-washu']
for sample in sr_samples:
    file_path = f'../../../mutect2/smaht/illumina/hapmap/mosdepth/{sample}.mosdepth.summary.txt'
    cov_df = pd.read_csv(file_path, sep='\t')
    cov = cov_df.loc[cov_df['chrom'] == 'chrM', 'mean'].item()
    comb_sr_cov.loc[len(comb_sr_cov)] = [sample, cov]
comb_sr_cov

comb_cov_df = pd.concat([comb_sr_cov, df[['Sample', 'Mito_Coverage']]])
comb_cov_df[['Donor', 'Seq_Tech', 'Center']] = comb_cov_df['Sample'].str.split('-', expand=True)
comb_cov_df = comb_cov_df.sort_values(['Center', 'Seq_Tech'])

comb_cov_df['Center'] = comb_cov_df['Center'].str.upper()

conditions = [
    (comb_cov_df['Seq_Tech'] == 'pacbio'),
    (comb_cov_df['Seq_Tech'] == 'ont'),
    (comb_cov_df['Seq_Tech'] == 'illumina'),
]

values = ['PacBio', 'ONT', 'Illumina']
comb_cov_df['Seq_Tech'] = np.select(conditions, values, default='Unknown')

comb_cov_df

In [ ]:
## plot mito coverage for hapmap illumina, pb, and ont 
sns.set_theme(style="ticks", context="talk", font_scale=0.75)
plt.figure(figsize=(6, 4))

sns.swarmplot(
    comb_cov_df,
    x="Seq_Tech",
    y="Mito_Coverage",
  #  style="Center",
    hue="Center",
    s=8,
    legend=True)

sns.boxplot(comb_cov_df,
            showmeans=True,
            meanline=True,
            meanprops={'color': 'k', 'ls': '-', 'lw': 2},
            medianprops={'visible': False},
            whiskerprops={'visible': False},
            zorder=10,
            x="Seq_Tech",
            y="Mito_Coverage",
            showfliers=False,
            showbox=False,
            showcaps=False)

plt.yscale('log')
plt.ylabel('mtDNA Coverage')
plt.xlabel('')

plt.savefig("plots/fig2-hapmap_coverage.pdf", dpi=300)
plt.show()


In [ ]:
## read in vcfeval results
df = pd.read_csv("../hapmap/hprc_truthset_eval/combined_tables/HPRC.vcfeval.by_sample.all.tsv", sep='\t')
df_long = pd.melt(df,
                  id_vars=['tool', 'tech', 'sample'],  
                  var_name='Metric',   
                  value_name='Value')

df_long['Metric'] = df_long['Metric'].replace({
    'precision': 'Precision',
    'sensitivity': 'Sensitivity',
    'f_measure': 'F1'
})

conditions = [
    (df_long['tech'] == 'pacbio'),
    (df_long['tech'] == 'ont'),
    (df_long['tech'] == 'illumina'),
]

values = ['PacBio', 'ONT', 'Illumina']
df_long['tech'] = np.select(conditions, values, default='Unknown')


conditions = [
    (df_long['tool'] == 'mitoscope'),
    (df_long['tool'] == 'himito'),
    (df_long['tool'] == 'mitorsaw'),
    (df_long['tool'] == 'mutect2'),

]

values = ['MitoScope', 'Himito', 'Mitorsaw', 'Mutect2']
df_long['tool'] = np.select(conditions, values, default='Unknown')


df_long 


In [ ]:
## plot precision, sensitivity, and F1 scores
sns.set_theme(style="whitegrid", context="talk", font_scale=0.75)

palette = sns.color_palette("Set2", n_colors=3)

g = sns.catplot(
    data=df_long[df_long['Metric'].isin(['Precision', 'Sensitivity', 'F1'])],
    x="tool",
    y="Value",
    col="tech",
    col_wrap=4,
    hue="Metric",
    kind="box",
    height=4,
    aspect=0.9,
    palette=palette,
    linewidth=1,
    fliersize=2,
    #legend_out=False,
    sharex=False,
    sharey=True,
    
)
sns.move_legend(g, "center right", bbox_to_anchor=(0.8,0.5), title="") 
#sns.move_legend(g, "lower center", ncol=3, title="") 

g.tick_params(axis='x', labelrotation=360)
g.set_titles("{col_name}", size=14, weight='bold')
g.set_axis_labels("", "Score", size=16)
g.set(ylim=(-0.05, 1))

for ax in g.axes.flat:
    sns.despine(ax=ax, left=False, right=False, bottom=False, top=False)
    ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.8)

    ticks = ax.get_xticks()
    if len(ticks) > 1:
        midpoints = [(ticks[i] + ticks[i+1]) / 2 for i in range(len(ticks)-1)]
        for x in midpoints:
            ax.axvline(
                x=x,
                color="gray",
                linestyle="--",
                linewidth=0.8,
                alpha=0.8,
                zorder=0,
            )
    
    for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)
            spine.set_color("black")

plt.savefig("plots/fig2-hapmap_sens_prec_f1.pdf", dpi=300)
plt.show()

In [ ]:
## read in vcfeval results for just SNVs and generate plots
df = pd.read_csv("../hapmap/hprc_truthset_eval/combined_tables/HPRC.vcfeval.by_sample.snvs.tsv", sep='\t')
df_long = pd.melt(df,
                  id_vars=['tool', 'tech', 'sample'],  
                  var_name='Metric',   
                  value_name='Value')

df_long['Metric'] = df_long['Metric'].replace({
    'precision': 'Precision',
    'sensitivity': 'Sensitivity',
    'f_measure': 'F1'
})

conditions = [
    (df_long['tech'] == 'pacbio'),
    (df_long['tech'] == 'ont'),
    (df_long['tech'] == 'illumina'),
]

values = ['PacBio', 'ONT', 'Illumina']
df_long['tech'] = np.select(conditions, values, default='Unknown')


conditions = [
    (df_long['tool'] == 'mitoscope'),
    (df_long['tool'] == 'himito'),
    (df_long['tool'] == 'mitorsaw'),
    (df_long['tool'] == 'mutect2'),

]

values = ['MitoScope', 'Himito', 'Mitorsaw', 'Mutect2']
df_long['tool'] = np.select(conditions, values, default='Unknown')


sns.set_theme(style="whitegrid", context="talk", font_scale=0.75)

palette = sns.color_palette("Set2", n_colors=3)

g = sns.catplot(
    data=df_long[df_long['Metric'].isin(['Precision', 'Sensitivity', 'F1'])],
    x="tool",
    y="Value",
    col="tech",
    col_wrap=4,
    hue="Metric",
    kind="box",
    height=4,
    aspect=0.9,
    palette=palette,
    linewidth=1,
    fliersize=2,
    #legend_out=False,
    sharex=False,
    sharey=True,
    
)
sns.move_legend(g, "center right", bbox_to_anchor=(0.8,0.5), title="") 
#sns.move_legend(g, "lower center", ncol=3, title="") 

g.tick_params(axis='x', labelrotation=360)
g.set_titles("{col_name}", size=14, weight='bold')
g.set_axis_labels("", "Score", size=16)
g.set(ylim=(-0.05, 1))

for ax in g.axes.flat:
    sns.despine(ax=ax, left=False, right=False, bottom=False, top=False)  # keep full axes box
    ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.8)

    # Get tick positions (x coordinates of categories)
    ticks = ax.get_xticks()

    # Compute midpoints between categories
    if len(ticks) > 1:
        midpoints = [(ticks[i] + ticks[i+1]) / 2 for i in range(len(ticks)-1)]
        for x in midpoints:
            ax.axvline(
                x=x,
                color="gray",
                linestyle="--",
                linewidth=0.8,
                alpha=0.8,
                zorder=0,   # keeps lines behind boxes
            )

    
    for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)
            spine.set_color("black")

#plt.savefig("plots/fig2-hapmap_sens_prec_f1.pdf", dpi=300)
plt.show()

g = sns.catplot(
    data=df_long[df_long['Metric'].isin(['true_positives_call', 'false_negatives', 'false_positives'])],
    x="tool",
    y="Value",
    col="tech",
    col_wrap=4,
    hue="Metric",
    hue_order=['true_positives_call', 'false_negatives', 'false_positives'],
    kind="box",
    height=4,
    aspect=0.9,
    palette=palette,
    linewidth=1,
    fliersize=2,
    #legend_out=False,
    sharex=False,
    sharey=True,
    
)
sns.move_legend(g, "center right", bbox_to_anchor=(0.8,0.5), title="") 
#sns.move_legend(g, "lower center", ncol=3, title="") 

g.tick_params(axis='x', labelrotation=360)
g.set_titles("{col_name}", size=14, weight='bold')
g.set_axis_labels("", "Score", size=16)
#g.set(ylim=(-0.05, 1))

for ax in g.axes.flat:
    sns.despine(ax=ax, left=False, right=False, bottom=False, top=False)
    ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.8)

    ticks = ax.get_xticks()
    if len(ticks) > 1:
        midpoints = [(ticks[i] + ticks[i+1]) / 2 for i in range(len(ticks)-1)]
        for x in midpoints:
            ax.axvline(
                x=x,
                color="gray",
                linestyle="--",
                linewidth=0.8,
                alpha=0.8,
                zorder=0
            )

    for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)
            spine.set_color("black")

#plt.savefig("plots/fig2-hapmap_sens_prec_f1.pdf", dpi=300)
plt.show()

In [ ]:
## read in vcfeval results for just indels and generate plots
df = pd.read_csv("../hapmap/hprc_truthset_eval/combined_tables/HPRC.vcfeval.by_sample.indels.tsv", sep='\t')
df_long = pd.melt(df,
                  id_vars=['tool', 'tech', 'sample'],  
                  var_name='Metric',   
                  value_name='Value')

df_long['Metric'] = df_long['Metric'].replace({
    'precision': 'Precision',
    'sensitivity': 'Sensitivity',
    'f_measure': 'F1'
})

conditions = [
    (df_long['tech'] == 'pacbio'),
    (df_long['tech'] == 'ont'),
    (df_long['tech'] == 'illumina'),
]

values = ['PacBio', 'ONT', 'Illumina']
df_long['tech'] = np.select(conditions, values, default='Unknown')


conditions = [
    (df_long['tool'] == 'mitoscope'),
    (df_long['tool'] == 'himito'),
    (df_long['tool'] == 'mitorsaw'),
    (df_long['tool'] == 'mutect2'),

]

values = ['MitoScope', 'Himito', 'Mitorsaw', 'Mutect2']
df_long['tool'] = np.select(conditions, values, default='Unknown')


sns.set_theme(style="whitegrid", context="talk", font_scale=0.75)
palette = sns.color_palette("Set2", n_colors=3)

g = sns.catplot(
    data=df_long[df_long['Metric'].isin(['Precision', 'Sensitivity', 'F1'])],
    x="tool",
    y="Value",
    col="tech",
    col_wrap=4,
    hue="Metric",
    kind="box",
    height=4,
    aspect=0.9,
    palette=palette,
    linewidth=1,
    fliersize=2,
    #legend_out=False,
    sharex=False,
    sharey=True,
    
)
sns.move_legend(g, "center right", bbox_to_anchor=(0.8,0.5), title="") 
#sns.move_legend(g, "lower center", ncol=3, title="") 

g.tick_params(axis='x', labelrotation=360)
g.set_titles("{col_name}", size=14, weight='bold')
g.set_axis_labels("", "Score", size=16)
g.set(ylim=(-0.05, 1))

for ax in g.axes.flat:
    sns.despine(ax=ax, left=False, right=False, bottom=False, top=False)
    ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.8)

    ticks = ax.get_xticks()
    if len(ticks) > 1:
        midpoints = [(ticks[i] + ticks[i+1]) / 2 for i in range(len(ticks)-1)]
        for x in midpoints:
            ax.axvline(
                x=x,
                color="gray",
                linestyle="--",
                linewidth=0.8,
                alpha=0.8,
                zorder=0
            )

    for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)
            spine.set_color("black")

#plt.savefig("plots/fig2-hapmap_sens_prec_f1.pdf", dpi=300)
plt.show()


g = sns.catplot(
    data=df_long[df_long['Metric'].isin(['true_positives_call', 'false_negatives', 'false_positives'])],
    x="tool",
    y="Value",
    col="tech",
    col_wrap=4,
    hue="Metric",
    hue_order=['true_positives_call', 'false_negatives', 'false_positives'],
    kind="box",
    height=4,
    aspect=0.9,
    palette=palette,
    linewidth=1,
    fliersize=2,
    #legend_out=False,
    sharex=False,
    sharey=True,
    
)
sns.move_legend(g, "center right", bbox_to_anchor=(0.8,0.5), title="") 
#sns.move_legend(g, "lower center", ncol=3, title="") 

g.tick_params(axis='x', labelrotation=360)
g.set_titles("{col_name}", size=14, weight='bold')
g.set_axis_labels("", "Score", size=16)
#g.set(ylim=(-0.05, 1))

for ax in g.axes.flat:
    sns.despine(ax=ax, left=False, right=False, bottom=False, top=False)
    ax.grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.8)

    ticks = ax.get_xticks()
    if len(ticks) > 1:
        midpoints = [(ticks[i] + ticks[i+1]) / 2 for i in range(len(ticks)-1)]
        for x in midpoints:
            ax.axvline(
                x=x,
                color="gray",
                linestyle="--",
                linewidth=0.8,
                alpha=0.8,
                zorder=0
            )

    for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)
            spine.set_color("black")

#plt.savefig("plots/fig2-hapmap_sens_prec_f1.pdf", dpi=300)
plt.show()

In [ ]:
## add mutect2 HF results to df
mutect_df = pd.read_csv('../../../mutect2/smaht/illumina/hapmap/output/merged.mutect2.filtered.norm.vcfeval.vcf.gz', 
                 comment='#', sep='\t', compression='gzip', names=['chrom', 'pos', 'id', 'ref', 'alt', 'qual', 'filter', 'info', 'format', 'sample'])

mutect_df['id'] = mutect_df[['chrom', 'pos', 'ref', 'alt']].astype(str).agg('-'.join, axis=1)
mutect_df['hp_mutect'] = mutect_df['info'].str.extract(r'AVG_HET=([0-9.]+);').astype(float)
mutect_df['tool'] = 'mutect2'
mutect_df['tech'] = 'illumina'
mutect_df = mutect_df[['id', 'hp_mutect']]
mutect_df


In [ ]:
## add mitoscope pb HF results to df
mitoscope_pb_df = pd.read_csv('../hapmap/pacbio/output/merged.baldur.vcfeval.vcf.gz', 
                 comment='#', sep='\t', compression='gzip', names=['chrom', 'pos', 'id', 'ref', 'alt', 'qual', 'filter', 'info', 'format', 'sample'])

mitoscope_pb_df['id'] = mitoscope_pb_df[['chrom', 'pos', 'ref', 'alt']].astype(str).agg('-'.join, axis=1)
mitoscope_pb_df['hp_mito_pb'] = mitoscope_pb_df['info'].str.extract(r'AVG_HET=([0-9.]+);').astype(float)
mitoscope_pb_df['tool'] = 'mitoscope'
mitoscope_pb_df['tech'] = 'pacbio'
mitoscope_pb_df = mitoscope_pb_df[['id', 'hp_mito_pb']]

mitoscope_pb_df

In [ ]:
## add mitoscope ont HF results to df
mitoscope_ont_df = pd.read_csv('../hapmap/ont/output/merged.baldur.vcfeval.vcf.gz', 
                 comment='#', sep='\t', compression='gzip', names=['chrom', 'pos', 'id', 'ref', 'alt', 'qual', 'filter', 'info', 'format', 'sample'])

mitoscope_ont_df['id'] = mitoscope_ont_df[['chrom', 'pos', 'ref', 'alt']].astype(str).agg('-'.join, axis=1)
mitoscope_ont_df['hp_mito_ont'] = mitoscope_ont_df['info'].str.extract(r'AVG_HET=([0-9.]+);').astype(float)
mitoscope_ont_df['tool'] = 'mitoscope'
mitoscope_ont_df['tech'] = 'ont'
mitoscope_ont_df = mitoscope_ont_df[['id', 'hp_mito_ont']]

mitoscope_ont_df

In [ ]:
## get list of variants in hprc truthset
hprc_df = pd.read_csv('../hapmap/hprc_truthset_eval/hprc-v2.0-mc-grch38.wave.vcfeval.vcf.gz', 
                 comment='#', sep='\t', compression='gzip', names=['chrom', 'pos', 'id', 'ref', 'alt', 'qual', 'filter', 'info', 'format', 'sample'])

hprc_df['id'] = hprc_df[['chrom', 'pos', 'ref', 'alt']].astype(str).agg('-'.join, axis=1)

truth_variants = hprc_df['id'].to_list()

In [ ]:
## combine individual HF dfs into merged one
merged_df = pd.merge(mutect_df, mitoscope_pb_df, on="id", how="inner")
final_merged_df = pd.merge(merged_df, mitoscope_ont_df, on="id", how="inner")

#final_merged_df = final_merged_df[(final_merged_df['id'].isin(truth_variants))]

final_merged_df['ref'] = final_merged_df['id'].str.split("-", expand=True)[2]
final_merged_df['alt'] = final_merged_df['id'].str.split("-", expand=True)[3]

final_merged_df['indel'] = final_merged_df.apply(lambda row: 'indel' if (len(row['ref']) > 1 or len(row['alt']) > 1) else 'snv', axis=1)
final_merged_df = final_merged_df.drop(columns=['ref','alt'])

final_merged_df['pos'] = final_merged_df['id'].str.split('-', expand=True)[1].astype(int)
final_merged_df = final_merged_df[~(final_merged_df['pos'].isin(poly_homopolymer_regions))]
final_merged_df = final_merged_df.drop(columns=["pos"])

final_merged_df

In [ ]:
## pearson correlation values
corr = final_merged_df.drop(columns=['id', 'indel']).corr()
corr

In [ ]:
## heteroplasmy concordance linear regreggion plots
sns.set_theme(style="ticks", context="talk", font_scale=1)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

common_kws = dict(scatter_kws={'s': 40, 'alpha': 0.7, 'edgecolor': 'black'},
                  line_kws={'color': 'red', 'lw': 2})

sns.regplot(data=final_merged_df, x='hp_mutect', y='hp_mito_pb', ax=axes[0], **common_kws)
axes[0].set_xlabel('Mutect2 Heteroplasmy')
axes[0].set_ylabel('MitoScope-PB Heteroplasmy')
axes[0].text(0.05, 0.95, f"r = {corr.loc['hp_mutect', 'hp_mito_pb']:.3f}",
             transform=axes[0].transAxes,
             ha="left", va="top")

sns.regplot(data=final_merged_df, x='hp_mutect', y='hp_mito_ont', ax=axes[1], **common_kws)
axes[1].set_xlabel('Mutect2 Heteroplasmy')
axes[1].set_ylabel('MitoScope-ONT Heteroplasmy')
axes[1].text(0.05, 0.95, f"r = {corr.loc['hp_mutect', 'hp_mito_ont']:.3f}",
             transform=axes[1].transAxes,
             ha="left", va="top")

sns.regplot(data=final_merged_df, x='hp_mito_pb', y='hp_mito_ont', ax=axes[2], **common_kws)
axes[2].set_xlabel('MitoScope-PB Heteroplasmy')
axes[2].set_ylabel('MitoScope-ONT Heteroplasmy')
axes[2].text(0.05, 0.95, f"r = {corr.loc['hp_mito_pb', 'hp_mito_ont']:.3f}",
             transform=axes[2].transAxes,
             ha="left", va="top")

plt.savefig("plots/fig2-hapmap_heteroplasmy_correlation_plots.pdf", dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
## heteroplasmy concordance heatmap
sns.set_theme(style="whitegrid", context="talk", font_scale=0.3)

labels = ['mutect2', 'mitoScope - PB', 'mitoScope - ONT']
df_sorted = final_merged_df.sort_values("hp_mutect", ascending=False).drop(columns=['indel'])

plt.figure(figsize=(5, 20))  # flip aspect ratio

ax = sns.heatmap(
    df_sorted.set_index(["id"]),
    xticklabels=labels,
    annot=True,
    cmap="viridis",
    linewidths=0.1,
    fmt=".3f"
)

plt.title("")
plt.tight_layout()
plt.savefig("plots/fig2-hapmap_heteroplasmy_concordance_heatmap.pdf", dpi=300)
plt.show()

df_sorted.to_csv('tables/hapmap_heteroplasmy_concordance.csv', index=False)


In [ ]:
## pileup verification for discordant positions (8860,4769)
positions = [8860, 4769]
indir="pileup_bases"

comb_x = pd.DataFrame()
for s in df['sample'].unique():
    for p in positions:
        x = pd.read_csv(f'{indir}/{s}.{p}.base_comp.txt', sep=" ", names=['freq', 'base'])
        x['sample'] = s
        x['pos'] = p
        comb_x = pd.concat([comb_x, x])
    
comb_x['base'] = comb_x['base'].str.upper()
#comb_x = comb_x[~comb_x['base'].isin(['*'])]

comb_x_collapsed = comb_x.groupby(['sample', 'pos', 'base'])['freq'].sum().reset_index()
comb_x_collapsed['prop'] = comb_x_collapsed['freq'] / comb_x_collapsed.groupby(['sample', 'pos'])['freq'].transform('sum')
comb_x_collapsed[['donor', 'tech', 'center']] = comb_x_collapsed['sample'].str.split('-', expand=True)
comb_x_collapsed

In [ ]:
## pileup results for the 2 discordant SNVs
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

g = sns.catplot(
    data=comb_x_collapsed,
    x='base',
    order=['A', 'G', 'T', 'C', '*'],
    y='prop',
    hue='tech',
    col='pos',
   # col='tech',
    kind='swarm',
    dodge=True,
    height=4,
    aspect=1,
    alpha=0.8,
    s=50
)

g.set_axis_labels("Base", "Proportion")
plt.savefig(f"plots/supplementary/pileup_freqs_discordant_variants.pdf", dpi=300)
plt.show()
